In [ ]:
import os

os.chdir("../")

In [ ]:
import torch
import fitz
import io
from PIL import Image
from transformers import AutoModelForTokenClassification, AutoProcessor
from collections import Counter

In [ ]:
pdf_path = "path/to/your_invoice.pdf"

In [ ]:
def get_page_object(uploaded_file, dpi=300):
    doc = fitz.open(uploaded_file)
    page = doc[0]
    page_area = page.rect.width * page.rect.height
    adaptive_dpi = min(dpi, max(150, int(300000 / page_area * 72)))
    mat = fitz.Matrix(adaptive_dpi / 72, adaptive_dpi / 72)
    pix = page.get_pixmap(matrix=mat)
    img_data = pix.tobytes("png")
    image = Image.open(io.BytesIO(img_data))

    return image

In [ ]:
image = get_page_object(pdf_path)

In [ ]:
def aggregate_page_data(uploaded_file):
    doc = fitz.open(uploaded_file)
    total_pages = len(doc)

    file_data = {"csv_entry": {}, "pages": {}}

    for i in range(total_pages):
        page = doc.load_page(i)

        words_data = page.get_text("words")

        file_data["pages"][i + 1] = {
            "words_data": words_data,
            "annotations": {},
            "pdf_width": page.rect.width,
            "pdf_height": page.rect.height,
        }

    doc.close()
    return file_data

In [ ]:
file_data = aggregate_page_data(pdf_path)

In [ ]:
page_data = file_data["pages"][1]

In [ ]:
raw_words_data = page_data["words_data"]

In [ ]:
pdf_width = page_data["pdf_width"]
pdf_height = page_data["pdf_height"]

In [ ]:
words = []
bboxes = []
for i, entry in enumerate(raw_words_data):
    words.append(raw_words_data[i][4])
    bbox = [
        raw_words_data[i][0],
        raw_words_data[i][1],
        raw_words_data[i][2],
        raw_words_data[i][3],
    ]
    bboxes.append(bbox)

In [ ]:
def normalize_box(bbox, width, height):
    return [
        int(1000 * (bbox[0] / width)),
        int(1000 * (bbox[1] / height)),
        int(1000 * (bbox[2] / width)),
        int(1000 * (bbox[3] / height)),
    ]

In [ ]:
def normalize_bboxes(bboxes, width, height):
    new_boxes = []
    for i, box in enumerate(bboxes):
        transformed = normalize_box(box, width, height)
        new_boxes.append(transformed)
    return new_boxes

In [ ]:
normalized_bboxes = normalize_bboxes(bboxes, pdf_width, pdf_height)
normalized_bboxes

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "app/machine_learning/separated/layoutlmv3-invoice-model-separated-training_args/checkpoint-300"
)

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

In [ ]:
encoding = processor(
    image,
    words,
    boxes=normalized_bboxes,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=512,
)

In [ ]:
with torch.no_grad():
    outputs = model(**encoding)

In [ ]:
logits = outputs.logits
predictions = logits.argmax(-1).squeeze().tolist()

In [ ]:
id2label = model.config.id2label

In [ ]:
predicted_labels = [id2label[p] for p in predictions]

In [ ]:
word_ids = encoding.word_ids()

In [ ]:
import torch.nn.functional as F

probs = F.softmax(logits, dim=-1)
confidences, _ = probs.max(dim=-1)
confidences = confidences.squeeze().tolist()

In [ ]:
final_confidences = []

for i, word_id in enumerate(word_ids):
    if word_id is None:
        continue
    if i == 0 or word_id != word_ids[i - 1]:
        final_confidences.append(confidences[i])

In [ ]:
final_labels = []
final_boxes = []
final_words = []

for i, word_id in enumerate(word_ids):
    if word_id is None:
        continue  # skip special/padding tokens

    # If this token is the first subtoken of a word
    if i == 0 or word_id != word_ids[i - 1]:
        final_labels.append(id2label[predictions[i]])
        final_boxes.append(encoding.bbox.squeeze().tolist()[i])
        final_words.append(words[word_id])

In [ ]:
for word, label, conf in zip(final_words, final_labels, final_confidences):
    print(f"{word:20s}  →  {label:30s}  ({conf:.3f})")

In [ ]:
token_boxes = encoding.bbox.squeeze().tolist()
width, height = image.size

In [ ]:
logits

In [ ]:
import cv2
import numpy as np
from PIL import ImageDraw, ImageFont

draw = ImageDraw.Draw(image)

for word, label, box in zip(final_words, final_labels, final_boxes):
    if label != "O":  # only draw entities
        x0, y0, x1, y1 = [
            box[0] * image.width / 1000,
            box[1] * image.height / 1000,
            box[2] * image.width / 1000,
            box[3] * image.height / 1000,
        ]
        draw.rectangle([x0, y0, x1, y1], outline="red", width=2)
        draw.text((x0, y0 - 10), label, fill="red")

image.show()

In [ ]:
from collections import defaultdict


def get_top_label_predictions(final_words, final_labels, final_confidences, top_n=3):
    # Group by label type (ignoring "O" and BIO prefixes)
    grouped = defaultdict(list)

    for word, label, conf in zip(final_words, final_labels, final_confidences):
        if label == "O":
            continue  # skip non-entity
        # Remove BIO prefix (B-, I-, S-) to group entities
        base_label = label.split("-", 1)[-1] if "-" in label else label
        grouped[base_label].append((word, label, conf))

    # Sort and select top N per label
    top_predictions = {}
    for base_label, items in grouped.items():
        items_sorted = sorted(items, key=lambda x: x[2], reverse=True)
        top_predictions[base_label] = items_sorted[:top_n]

    return top_predictions

In [ ]:
top_preds = get_top_label_predictions(
    final_words, final_labels, final_confidences, top_n=4
)

In [ ]:
for label, entries in top_preds.items():
    print(f"\n🔹 {label}")
    for word, full_label, conf in entries:
        print(f"   {word:25s} ({full_label:20s})  conf={conf:.3f}")